In [ ]:
# ============================================
# CELL 1: Install Dependencies
# ============================================
!pip install -q openai tiktoken pandas numpy python-dotenv tenacity spacy




In [ ]:


# ============================================
# CELL 2: Mount Google Drive & Define Paths
# ============================================
from google.colab import drive
drive.mount('/content/drive')

# ---- BASE PATHS ----
BASE = "/content/drive/MyDrive/PropInsight"   # your project base folder

# Raw inputs (point to your raw gov folder)
RAW_BASE = f"{BASE}/raw/government_websites"  # e.g., .../raw/government_websites/**/*.json[l]

# Normalized output (we'll create this)
NORMALIZED_DIR = f"{BASE}/normalized"
RAW_COMBINED   = f"{NORMALIZED_DIR}/government_raw_combined.csv"   # will be created by this notebook

# Fallback processed inputs (if RAW missing)
PROCESSED_BASE = f"{BASE}/processed"  # per-agency fallback

# Labeled outputs
OUTPUT_DIR = f"{BASE}/labeled"        # where *_labeled.csv and *_labeled_enriched.csv will go

# Domain resources
SINGLEX_CSV = f"{BASE}/corpus/Singlish/lexicon.csv"
ENTITYRULER = f"{BASE}/corpus/SGPropertyDomain/spacy_entityruler_patterns.jsonl"
REGEX_JSONL = f"{BASE}/corpus/SGPropertyDomain/regex_patterns.jsonl"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# CELL 4: Config (Models & Run Options)
# ============================================
MODEL        = "gpt-4o"  # "gpt-4o" for higher quality
BATCH_SIZE   = 8              # gentle batching
MAX_RECORDS  = 0              # 0 = all rows; set small number for smoke test
AGENCY_ONLY  = ""             # e.g., "HDB" to filter


In [ ]:

# ============================================
# CELL 5: Imports & Shared Helpers
# ============================================
import re, json, time, logging, pandas as pd, numpy as np, hashlib, glob
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple
from datetime import datetime
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from pathlib import Path
from openai import OpenAI

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("propinsight")

def clean_text(s: Any) -> str:
    if s is None: return ""
    s = str(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def safe_json_load(s: Any):
    if not isinstance(s, str): return None
    s = s.strip()
    if not s: return None
    try:
        return json.loads(s)
    except Exception:
        try:
            return json.loads(re.sub(r"[‘’“”]", '"', s))
        except Exception:
            return None

def pick_val(d):
    if isinstance(d, dict):
        for k in ["value","sentiment","classification","label","tone"]:
            if k in d and d[k]:
                return str(d[k]).lower()
    if isinstance(d, str): return d.lower()
    return None

def pick_score(d):
    if isinstance(d, dict):
        for k in ["confidence_score","confidence","score","overall_sentiment_confidence","overall_sentiment_confidence_score"]:
            if k in d:
                try: return float(d[k])
                except: pass
    return None

def now_iso():
    from datetime import timezone
    return datetime.now(timezone.utc).isoformat()


In [ ]:


# ============================================
# CELL 6: Gazetteer, Policy Lexicon, Singlish Lexicon
# ============================================
GAZETTEER = [
    "Ang Mo Kio","Bishan","Bukit Batok","Bukit Panjang","Bukit Timah",
    "Choa Chu Kang","Clementi","Geylang","Hougang","Jurong East","Jurong West",
    "Kallang","Marine Parade","Pasir Ris","Punggol","Queenstown","Sembawang",
    "Sengkang","Serangoon","Tampines","Toa Payoh","Woodlands","Yishun",
    "Bedok","Boon Lay","Orchard","Marina Bay","Novena","Tanglin","River Valley",
    "Seletar","Bukit Merah","Kallang Basin","Braddell","MacPherson",
    "Outram","Telok Blangah","Downtown Core","Harbourfront","Balestier",
    "Central Region","East Region","North Region","North-East Region","West Region",
    "District 1 - Raffles Place, Cecil, Marina, People's Park",
    "District 2 - Anson, Tanjong Pagar",
    "District 3 - Queenstown, Tiong Bahru",
    "District 4 - Telok Blangah, Harbourfront",
    "District 5 - Pasir Panjang, Hong Leong Garden, Clementi",
    "District 6 - High Street, Beach Road (part)",
    "District 7 - Middle Road, Golden Mile",
    "District 8 - Little India, Farrer Park",
    "District 9 - Orchard, Cairnhill, River Valley",
    "District 10 - Bukit Timah, Holland Road, Tanglin",
    "District 11 - Watten Estate, Novena, Thomson",
    "District 12 - Balestier, Toa Payoh, Serangoon",
    "District 13 - MacPherson, Braddell",
    "District 14 - Geylang, Eunos",
    "District 15 - Katong, Joo Chiat, Amber Road",
    "District 16 - Bedok, Upper East Coast, Eastwood, Kew Drive",
    "District 17 - Loyang, Changi",
    "District 18 - Tampines, Pasir Ris",
    "District 19 - Serangoon Garden, Hougang, Punggol",
    "District 20 - Bishan, Ang Mo Kio",
    "District 21 - Upper Bukit Timah, Clementi Park, Ulu Pandan",
    "District 22 - Jurong (East and West)",
    "District 23 - Hillview, Dairy Farm, Bukit Panjang, Choa Chu Kang",
    "District 24 - Lim Chu Kang, Tengah",
    "District 25 - Kranji, Woodgrove",
    "District 26 - Upper Thomson, Springleaf",
    "District 27 - Yishun, Sembawang",
    "District 28 - Seletar"
]

POLICY_LEXICON = [
    "URA Master Plan", "Concept Plan", "Long-Term Plan", "Draft Master Plan 2025",
    "Planning Area", "Subzone", "Development Control", "Land Use Zoning",
    "Zoning Regulations", "Plot Ratio", "Gross Plot Ratio", "GFA", "Gross Floor Area",
    "Site Coverage", "Building Height Plan", "Building Height Control",
    "Special and Detailed Control Plans", "Landed Housing Areas Plan",
    "Mixed-Use Development", "White Site", "Regional Centre",
    "Decentralisation Strategy", "Rejuvenation Corridor", "Transit-Oriented Development",
    "Underground Master Plan", "Land Reclamation", "URA Space",
    "URA Circular", "URA Planning Guidelines", "URA Planning Permission",
    "Development Control Rules", "Permissible Land Use",
    "BTO", "Build-to-Order", "HDB", "Housing and Development Board",
    "Public Housing", "Public Housing Policy", "Resale Flat", "Sale of Balance Flats",
    "Executive Condominium", "EC", "DBSS", "Design Build and Sell Scheme",
    "Prime Location Public Housing", "PLPH", "Plus Model", "Standard Model",
    "Public Scheme", "Fiancé Fiancée Scheme", "Orphans Scheme",
    "Single Singapore Citizen Scheme", "Joint Singles Scheme",
    "Non-Citizen Spouse Scheme", "Non-Citizen Family Scheme", "Conversion Scheme",
    "Family and Parenthood Priority Scheme", "FPPS",
    "Married Child Priority Scheme", "MCPS", "Family Care Scheme", "FCS",
    "Third Child Priority Scheme", "Proximity Priority Scheme",
    "Selective En Bloc Redevelopment Scheme", "SERS",
    "Home Improvement Programme", "HIP", "Neighbourhood Renewal Programme", "NRP",
    "Remaking Our Heartland", "ROH", "Lift Upgrade Programme", "LUP",
    "En Bloc", "Collective Sale",
    "CPF Housing Grant", "Enhanced CPF Housing Grant", "EHG",
    "Proximity Housing Grant", "PHG", "Step-Up CPF Housing Grant", "SUHG",
    "Family Grant", "Half Housing Grant", "Singles Grant",
    "Fresh Start Housing Scheme", "Deferred Income Assessment",
    "Staggered Downpayment Scheme",
    "Lease Buyback Scheme", "LBS", "Silver Housing Bonus",
    "HDB Concessionary Loan", "HDB Loan", "CPF Housing Loan",
    "Resale Levy", "HDB Fire Insurance",
    "Income Ceiling", "Minimum Occupation Period", "MOP",
    "Private Condominium", "Leasehold", "Freehold", "99-Year Lease", "999-Year Lease",
    "Strata Title", "Strata-Titled", "Land Title",
    "Good Class Bungalow", "GCB", "Landed Property", "Terrace House",
    "Semi-Detached", "Bungalow", "Shophouse", "Conservation Shophouse",
    "HUDC", "Housing and Urban Development Company",
    "Conservation Area", "Conservation Building", "Conserved Property",
    "Heritage Building", "Heritage and Identity Plan",
    "Preservation Scheme", "Preservation of Monuments Act",
    "National Monument", "Preservation of Monuments Board",
    "URA Conservation Guidelines", "Built Heritage", "Adaptive Re-use",
    "Planning Act", "Land Acquisition Act", "State Lands Act",
    "Land Titles Strata Act", "LTSA", "Building Maintenance and Strata Management Act", "BMSMA",
    "Building Control Act", "Street Works Act",
    "Temporary Occupation Permit", "TOP", "Certificate of Statutory Completion", "CSC",
    "Written Permission", "Grant of Written Permission",
    "Differential Premium", "Land Betterment Charge",
    "Statutory Land Grant", "Government Land Grant",
    "Ministry of National Development", "MND",
    "Ministry of Law", "MinLaw", "Ministry of Trade and Industry", "MTI",
    "Urban Redevelopment Authority", "URA",
    "Housing and Development Board", "HDB",
    "Building and Construction Authority", "BCA",
    "Land Transport Authority", "LTA",
    "National Parks Board", "NParks",
    "Singapore Land Authority", "SLA",
    "Jurong Town Corporation", "JTC",
    "Council for Estate Agencies", "CEA",
    "Central Provident Fund", "CPF", "CPF Board",
    "Town Council", "Management Corporation Strata Title", "MCST",
    "Real Estate Developers Association of Singapore", "REDAS",
    "Community Mediation Centre", "CMC",
    "Government Land Sales", "GLS", "Government Land Sales Programme",
    "Reserve List", "Confirmed List", "Commercial Redevelopment",
    "Industrial Development", "Business Park", "Business 1", "Business 2",
    "White Zone", "Business Park White",
    "Land Transport Master Plan", "LTMP", "Land Transport Master Plan 2040",
    "Sustainable Transport Strategy", "MRT", "Mass Rapid Transit",
    "LRT", "Light Rail Transit", "Bus Interchange",
    "Integrated Transport Hub", "Walk Cycle Ride Plus",
    "Green Plan 2030", "Singapore Green Plan",
    "Green Mark", "Green Mark Scheme", "BCA Green Mark",
    "Zero Waste Masterplan", "Coastal Protection Plan",
    "Nature Corridor", "Green Corridor", "Park Connector Network",
    "Urban Heat Island Mitigation", "Energy Efficiency Programme",
    "Sustainable Development", "Liveability Framework",
    "Central Business District", "CBD", "Downtown Core",
    "Ethnic Integration Policy", "EIP", "Singapore Permanent Resident", "SPR",
    "Singapore Citizen", "SC", "HDB Flat Eligibility", "HFE Letter",
    "Income Cap", "Household Income", "Average Gross Monthly Household Income",
    "First-Timer", "Second-Timer", "Resale Market",
    "Property Cooling Measures", "Additional Buyer's Stamp Duty", "ABSD",
    "Buyer's Stamp Duty", "BSD", "Seller's Stamp Duty", "SSD",
    "Central Region", "East Region", "North Region", "North-East Region", "West Region",
    "Mature Estate", "Non-Mature Estate", "Waterfront District",
    "Subsidiary Proprietor", "Common Property", "Accessory Lot",
    "3Gen Flat", "Multi-Generation Flat", "Studio Apartment",
    "2-Room Flexi", "Prime Area", "Core Central Region", "CCR",
    "Rest of Central Region", "RCR", "Outside Central Region", "OCR"
]

POLICY_KEYS = ["absd","tdsr","ssd","ltv","mop","bto","gls","cooling measures","hfe","phg","fresh start"]

def load_singlish_terms(csv_path: str) -> List[str]:
    p = Path(csv_path)
    if not p.exists(): return []
    try:
        df = pd.read_csv(p)
        for col in ["term","word","phrase","singlish","lexeme"]:
            if col in df.columns:
                return [str(x).strip() for x in df[col].dropna().unique().tolist() if str(x).strip()]
        values = []
        for col in df.columns:
            values.extend([str(x).strip() for x in df[col].dropna().tolist()])
        return sorted(set([v for v in values if v]))
    except Exception:
        return []

SINGLISH_TERMS = set(load_singlish_terms(SINGLEX_CSV) + [
    "lah","lor","meh","sia","hor","leh","shiok","steady","chope","kiasu","paiseh","sian"
])

def policy_flag(text: str) -> bool:
    t = (text or "").lower()
    return any(k in t for k in POLICY_KEYS)

def first_location(text: str) -> Optional[str]:
    t = (text or "").lower()
    for loc in GAZETTEER:
        if loc.lower() in t:
            return loc
    return None

POLICY_TERMS = sorted(set([t.strip() for t in POLICY_LEXICON if str(t).strip()]))
POLICY_TERMS_LOWER = [t.lower() for t in POLICY_TERMS]

def find_policy_terms(text: str, max_hits: int = 20) -> Tuple[List[str], int]:
    if not text: return [], 0
    hay = " " + re.sub(r"\s+", " ", text.lower()).strip() + " "
    hits = []
    for t in POLICY_TERMS_LOWER:
        patt = f" {t} "
        if patt in hay:
            hits.append(t)
            if len(hits) >= max_hits:
                break
    return hits, len(hits)

def enrich_with_policy_lexicon(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    hay = (df.get("title","").astype(str) + " " + df.get("clean_text","").astype(str))
    term_hits, term_counts = [], []
    for txt in hay.tolist():
        hits, cnt = find_policy_terms(txt, max_hits=20)
        term_hits.append("; ".join(hits) if hits else None)
        term_counts.append(int(cnt))
    df["policy_terms"] = term_hits
    df["policy_term_count"] = term_counts
    df["policy_mentioned"] = df.get("policy_mentioned", False) | (df["policy_term_count"] > 0)
    return df


In [ ]:

# ============================================
# CELL 7: RAW → NORMALIZED (build combined from *.json/*.jsonl)
# ============================================
def read_json_any(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".jsonl":
        rows = []
        with path.open("r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line = line.strip()
                if not line: continue
                try:
                    obj = json.loads(line)
                    if isinstance(obj, dict) and "articles" in obj and isinstance(obj["articles"], list):
                        for a in obj["articles"]:
                            if isinstance(a, dict):
                                a["_container_meta"] = {k:v for k,v in obj.items() if k!="articles"}
                                rows.append(a)
                    else:
                        rows.append(obj)
                except Exception:
                    pass
        df = pd.json_normalize(rows, max_level=2) if rows else pd.DataFrame()
        df["source_file"] = str(path.name)
        return df

    try:
        with path.open("r", encoding="utf-8", errors="ignore") as f:
            obj = json.load(f)
    except Exception as e:
        try:
            df = pd.read_json(str(path))
            df["source_file"] = str(path.name)
            return df
        except Exception:
            return pd.DataFrame({"_read_error":[str(e)], "source_file":[path.name]})

    if isinstance(obj, list):
        df = pd.json_normalize(obj, max_level=2); df["source_file"] = str(path.name); return df
    if isinstance(obj, dict):
        if "articles" in obj and isinstance(obj["articles"], list):
            df = pd.json_normalize(obj["articles"], max_level=2)
            meta = obj.get("scraping_metadata") or {}
            for k,v in meta.items():
                df[f"scrape_meta.{k}"] = json.dumps(v) if isinstance(v, (list,dict)) else v
            df["source_file"] = str(path.name); return df
        if "data" in obj and isinstance(obj["data"], list):
            df = pd.json_normalize(obj["data"], max_level=2); df["source_file"] = str(path.name); return df
        df = pd.json_normalize(obj, max_level=2); df["source_file"] = str(path.name); return df
    return pd.DataFrame(columns=["source_file"])

def infer_agency_from_path(p: Path) -> str:
    parts = [s.lower() for s in p.parts]
    for cand in ["hdb","mnd","mas","bca","sfa","sla"]:
        if cand in parts or p.name.lower().startswith(cand+"_"):
            return cand.upper()
    return "UNK"

def normalize_raw_to_combined(raw_glob_root: str, out_csv: str) -> pd.DataFrame:
    paths = glob.glob(f"{raw_glob_root}/**/*.json*", recursive=True)
    if not paths:
        raise FileNotFoundError(f"No JSON/JSONL found under {raw_glob_root}")
    frames=[]
    for s in sorted(paths):
        p = Path(s)
        df = read_json_any(p)
        if df.empty: continue
        agency = infer_agency_from_path(p)
        if "agency" not in df.columns:
            df.insert(0, "agency", agency)
        else:
            df["agency"] = df["agency"].fillna(agency).replace("", agency)
        for c in ["text","title","timestamp","url","language","id","source"]:
            if c not in df.columns: df[c] = df.get(c, None)
        frames.append(df)
    if not frames:
        raise RuntimeError("No usable rows parsed from raw JSON/JSONL.")
    df = pd.concat(frames, ignore_index=True, sort=False)
    df["title"] = df.get("metadata.title", df.get("title","")).apply(clean_text)
    df["text"]  = df.get("text","").apply(clean_text)
    df = df[(df["title"].astype(str).str.len()>0) | (df["text"].astype(str).str.len()>0)].copy()

    def row_key(r):
        if pd.notna(r.get("url")) and str(r["url"]).strip():
            return "url:" + str(r["url"]).strip().lower()
        key = (str(r.get("title","")) + "||" + str(r.get("text",""))).lower()
        return "tt:" + hashlib.md5(key.encode("utf-8")).hexdigest()
    df["_key"] = df.apply(row_key, axis=1)
    df = df.sort_values(by=["timestamp","source_file"], na_position="last").drop_duplicates("_key")
    df.drop(columns=["_key"], inplace=True, errors="ignore")
    df["timestamp"] = df.get("timestamp").astype(str)

    Path(NORMALIZED_DIR).mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False)
    logger.info(f"[NORMALIZE] Combined rows: {len(df)} saved → {out_csv}")
    return df

# ============================================
# CELL 7.1: Robust RAW discovery + processed fallback
# ============================================
def _count_json(root: str) -> int:
    return len(glob.glob(f"{root}/**/*.json*", recursive=True))

def find_best_raw_root(base: str) -> str:
    candidates = [
        f"{base}/raw/government_websites",
        f"{base}/raw",
        f"{base}/data/raw",
        f"{base}/government_websites",
        f"{base}/downloads",
        f"{base}/scraped",
    ]
    scored = [(c, _count_json(c)) for c in candidates]
    scored.sort(key=lambda x: x[1], reverse=True)
    print("[RAW SEARCH] Candidates & counts:")
    for c, n in scored: print(f"  - {c} : {n}")
    best, hits = scored[0]
    return best if hits > 0 else ""

def build_combined_from_processed(processed_root: str, out_csv: str) -> pd.DataFrame:
    frames = []
    for p in Path(processed_root).glob("*/*.csv"):
        try:
            df = pd.read_csv(p)
            if "agency" not in df.columns:
                df.insert(0, "agency", p.parent.name.upper())
            if "cleaned_text" in df.columns:
                df["text"] = df["cleaned_text"]
            if "title" not in df.columns and "metadata.title" in df.columns:
                df["title"] = df["metadata.title"]
            for c in ["id","timestamp","url","language","title","text"]:
                if c not in df.columns: df[c] = df.get(c, None)
            frames.append(df)
        except Exception as e:
            print(f"[WARN] Skipping {p} : {e}")
    if not frames:
        raise FileNotFoundError(f"No processed CSVs found under {processed_root}")
    df = pd.concat(frames, ignore_index=True, sort=False)
    df["title"] = df.get("title","").astype(str).fillna("").str.strip()
    df["text"]  = df.get("text","").astype(str).fillna("").str.strip()
    df = df[(df["title"].str.len()>0) | (df["text"].str.len()>0)].copy()

    def row_key(r):
        if pd.notna(r.get("url")) and str(r["url"]).strip():
            return "url:" + str(r["url"]).strip().lower()
        key = (str(r.get("title","")) + "||" + str(r.get("text",""))).lower()
        return "tt:" + hashlib.md5(key.encode("utf-8")).hexdigest()
    df["_key"] = df.apply(row_key, axis=1)
    df = df.sort_values(by=["timestamp"], na_position="last").drop_duplicates("_key")
    df.drop(columns=["_key"], inplace=True, errors="ignore")

    Path(NORMALIZED_DIR).mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False)
    print(f"[FALLBACK] Built combined from processed CSVs → {out_csv} (rows={len(df)})")
    return df





In [ ]:

# ============================================
# CELL 8: Optional Regex Flags + spaCy EntityRuler
# ============================================
def load_jsonl(path: Path):
    items=[]
    if not path.exists(): return items
    with path.open("r", encoding="utf-8") as f:
        for ln in f:
            ln=ln.strip()
            if ln:
                try: items.append(json.loads(ln))
                except: pass
    return items

def compile_regex_flags(regex_jsonl_path: str):
    pairs=[]
    p = Path(regex_jsonl_path)
    if not p.exists():
        logger.info(f"[INFO] No regex patterns at {p} — skipping rx_* flags.")
        return pairs
    for it in load_jsonl(p):
        pat = it.get("pattern"); name = it.get("name","pattern")
        if not pat:
            continue
        try:
            pairs.append((f"rx_{name}", re.compile(pat, flags=re.I)))
        except re.error:
            pass
    logger.info(f"[INFO] Loaded {len(pairs)} regex flags from {p.name}")
    return pairs

def apply_regex_flags(df: pd.DataFrame, pairs: List[tuple]) -> pd.DataFrame:
    if not pairs: return df
    tcol = next((c for c in ["clean_text","text","body"] if c in df.columns), None)
    if not tcol: return df
    series = df[tcol].astype(str)
    for colname, patt in pairs:
        try:
            df[colname] = series.str.contains(patt, na=False)
        except Exception:
            df[colname] = False
    return df

def entityruler_entities(df: pd.DataFrame, patterns_path: str) -> pd.DataFrame:
    try:
        import spacy
        p = Path(patterns_path)
        if not p.exists():
            logger.info(f"[INFO] No EntityRuler patterns at {p} — skipping.")
            if "entities" not in df.columns:
                df["entities"] = [[] for _ in range(len(df))]
            return df
        nlp = spacy.blank("en")
        ruler = nlp.add_pipe("entity_ruler")
        ruler.from_disk(str(p))
        tcol = next((c for c in ["clean_text","text","body"] if c in df.columns), None)
        if not tcol:
            df["entities"] = [[] for _ in range(len(df))]
            return df
        ents=[]
        for doc in nlp.pipe(df[tcol].astype(str).tolist(), batch_size=64):
            ents.append([{"text": e.text, "label": e.label_} for e in doc.ents])
        df["entities"] = ents
        df["entity"] = df["entities"].apply(lambda L: "; ".join(dict.fromkeys([e["text"] for e in L])) if isinstance(L,list) else None)
        df["entity_labels"] = df["entities"].apply(lambda L: "; ".join(dict.fromkeys([e["label"] for e in L])) if isinstance(L,list) else None)
        return df
    except Exception as e:
        logger.warning(f"[WARN] EntityRuler failed: {e}")
        if "entities" not in df.columns:
            df["entities"] = [[] for _ in range(len(df))]
        return df


In [ ]:
# ============================================
# CELL 9: GPT Labeler (prompts + resilient calls)
# ============================================
@dataclass
class GPTConfig:
    model: str = MODEL
    max_tokens: int = 850
    temperature: float = 0.0

def build_messages(prompt: str) -> List[Dict[str, str]]:
    return [
        {"role": "system", "content": "You are a Singapore real estate expert. ALWAYS reply with VALID JSON only."},
        {"role": "user", "content": prompt}
    ]

PROPERTY_SENTIMENT_PROMPT = """You are a Singapore real estate expert. Analyze the market tone of the text.
Return strict JSON:
overall_sentiment: "positive"|"neutral"|"negative" with confidence_score (0..1)
property_market_sentiment: "bullish"|"neutral"|"bearish" with confidence_score
investor_sentiment: "optimistic"|"neutral"|"pessimistic" with confidence_score
key_sentiment_drivers: list of short phrases
sentiment_intensity: integer (1..10)

TEXT:
{body}
"""

ASPECT_PROMPT = """Aspect-based sentiment for SG property. Aspects:
price_affordability, market_supply, demand_trends, government_policies, infrastructure_development, economic_factors.
Return strict JSON mapping each aspect -> {sentiment, confidence, key_phrases(list)}.
Sentiment in ["positive","neutral","negative","unknown","not_applicable"].

TEXT:
{body}
"""

SINGLISH_PROMPT = """Detect Singlish/cultural cues. Return strict JSON:
singlish_detected (true/false), singlish_terms (list), cultural_references (list),
local_context_score (0..10), formality_level ("formal"|"semi-formal"|"informal"), target_audience ("locals"|"expats"|"general").

TEXT:
{body}
"""

ENTITIES_PROMPT = """Extract entities (SG real estate). Return strict JSON:
organizations (list), locations (list), persons (list), policies (list), financial_figures (list), dates (list).
Only include if explicitly present.

TEXT:
{body}
"""

POLICY_IMPACT_PROMPT = """Policy implications (ABSD, TDSR, LTV, BTO, GLS, etc.). Return strict JSON:
policy_relevance: "high"|"medium"|"low",
policy_type: "regulatory"|"fiscal"|"monetary"|"planning"|"other",
affected_segments (list), impact_timeline, impact_magnitude, stakeholders_affected (list).

TEXT:
{body}
"""

LOCATION_IMPACT_PROMPT = """Location impacts. Return strict JSON:
locations_mentioned (list), location_sentiment (map name->"positive"|"neutral"|"negative"),
development_impact (list), accessibility_impact (list), property_value_implications (list).

TEXT:
{body}
"""

DOMAIN_EXPERTISE_PROMPT = """Expert assessment (succinct). Return strict JSON:
expertise_assessment, market_context, risk_factors (list), opportunities (list), recommendations (list).

TEXT:
{body}
"""

class GPTLabeler:
    def __init__(self, cfg: GPTConfig):
        api_key = os.environ.get("OPENAI_API_KEY", "")
        if not api_key:
            raise RuntimeError("OPENAI_API_KEY is not set.")
        self.client = OpenAI(api_key=api_key)
        self.cfg = cfg

    @retry(
        reraise=True,
        stop=stop_after_attempt(5),
        wait=wait_exponential(multiplier=1, min=1, max=20),
        retry=retry_if_exception_type(Exception),
    )
    def _json_call(self, prompt: str) -> Dict[str, Any]:
        resp = self.client.chat.completions.create(
            model=self.cfg.model,
            messages=build_messages(prompt),
            temperature=self.cfg.temperature,
            max_tokens=self.cfg.max_tokens,
            response_format={"type": "json_object"},
        )
        txt = (resp.choices[0].message.content or "").strip()
        if txt.startswith("```"):
            txt = txt.strip("`")
            txt = re.sub(r"^json\n", "", txt, flags=re.I)
        return json.loads(txt)

    def label_one(self, body: str) -> Dict[str, Any]:
        out = {}
        out["property_sentiment"]     = self._json_call(PROPERTY_SENTIMENT_PROMPT.format(body=body))
        out["aspect_based_sentiment"] = self._json_call(ASPECT_PROMPT.format(body=body))
        out["singlish_cultural"]      = self._json_call(SINGLISH_PROMPT.format(body=body))
        out["named_entities"]         = self._json_call(ENTITIES_PROMPT.format(body=body))
        out["policy_impact"]          = self._json_call(POLICY_IMPACT_PROMPT.format(body=body))
        out["location_impact"]        = self._json_call(LOCATION_IMPACT_PROMPT.format(body=body))
        out["domain_expertise"]       = self._json_call(DOMAIN_EXPERTISE_PROMPT.format(body=body))
        return out


In [ ]:

#============================================
# CELL 10: Cleaner + Heuristic Bootstrap + Flatteners/Flags
# ============================================
# ---- Cleaner ----
NOISE_PATTERNS = [
    r"\bAbout Us\b.*?\bPress Releases\b",
    r"\bNewsroom\b.*?\bPress Releases\b",
    r"\bSelect Year\b.*?(?:From\s*To\s*Go)?",
    r"\bRead press release\b",
    r"\bRight arrow icon\b", r"\bPrevNext\b",
    r"\bFind out more\b", r"\bLearn more\b",
    r"\bClick here\b", r"\bRead more\b",
    r"\bIssued\s+By:?.*", r"\bIssued\s+by\s+.*",
    r"\bAnnex(?:es)?[: ]?.*",
    r"\[Credit:.*?\]", r"\(Credit:.*?\)",
    r"\bShare this\b.*", r"\bFollow us\b.*",
    r"\bFAQs?\b", r"\bContact us\b",
    r"©\s*\d{4}.*",
]
ALLOWLIST = set([
    "BTO","ABSD","TDSR","LTV","HDB","SLA","URA","MND","MAS","BCA","SFA","GLS","PLPH","PHG","EHG","HIP","NRP",
    "Woodlands","Sembawang","Punggol","Tampines","CCR","RCR","OCR","Executive Condominium"
])

def strip_noise(text: str) -> str:
    if not isinstance(text, str): return ""
    t = text.replace("\u00a0"," ").replace("\u200b"," ").replace("…","...")
    for pat in NOISE_PATTERNS:
        t = re.sub(pat, " ", t, flags=re.I|re.S)
    t = re.sub(r"(?:\|\s*){2,}", " ", t)
    t = re.sub(r"(Read press release\s*)+", " ", t, flags=re.I)
    t = re.sub(r"\b(Prev|Next|Back|Top)\b", " ", t, flags=re.I)
    t = re.sub(r"(?:(?:[A-Z][a-z]+)\s*){6,}", " ", t)
    t = re.sub(r"\s{2,}", " ", t).strip()
    return t

def basic_sentence_keep(t: str) -> str:
    if not t: return t
    sents = re.split(r"(?<=[\.\!\?])\s+", t)
    kept = []
    for s in sents:
        s_stripped = s.strip()
        if not s_stripped:
            continue
        if (len(s_stripped) >= 40
            or any(tok in s_stripped for tok in ALLOWLIST)
            or re.search(r"\b\d{2,}\b", s_stripped)):
            kept.append(s_stripped)
    out = " ".join(kept)
    return re.sub(r"\s{2,}", " ", out).strip()

def clean_government_text(text: str) -> str:
    return basic_sentence_keep(strip_noise(text))

def apply_cleaner(df: pd.DataFrame) -> pd.DataFrame:
    title = df.get("metadata.title", df.get("title","")).astype(str).fillna("")
    raw = df.get("cleaned_text", df.get("text","")).astype(str).fillna("")
    df = df.copy()
    df["title"] = title.apply(lambda s: re.sub(r"\s{2,}"," ", s).strip())
    df["clean_text"] = raw.apply(clean_government_text)
    df = df[(df["title"].str.len()>0) | (df["clean_text"].str.len()>60)].copy()
    df["body"] = (df["title"].fillna("") + "\n\n" + df["clean_text"].fillna("")).str.strip()
    return df

# ---- Heuristic Bootstrap ----
POS_CUES = [
    r"\blaunch(ed|es|ing)?\b", r"\baward(ed|s)?\b", r"\bupgrade(d|s|ing)?\b",
    r"\bextend(ed|s|ing)? lease\b", r"\bimprov(e|ed|es|ing)\b", r"\benhanc(e|ed|es|ing)\b",
    r"\btime and cost savings\b", r"\bstabilis(?:e|ation)\b", r"\bbenefit(s|ted)?\b",
    r"\bhighest since\b", r"\breduce(d|s)? congestion\b", r"\bclimate resilience\b",
]
NEG_CUES = [
    r"\brecall(ed|s|ing)?\b", r"\bsinkhole\b", r"\bincident\b", r"\bdelay(ed|s|ing)?\b",
    r"\binvestigation\b", r"\bfraud\b", r"\bfatal\b", r"\bcontravention\b",
    r"\bgazetted for acquisition\b", r"\bnon-compliance\b",
]
NEU_CUES = [
    r"\bprovisional tender result(s)?\b", r"\btender(s)? close(d)?\b",
    r"\bpress release\b", r"\bnewsroom\b", r"\bview\b", r"\bannex\b",
    r"\bissued by\b", r"\bselect year\b", r"\bfrom to\b",
]
ASPECT_CUES = {
    "price_affordability":  [r"\bprice(s)? (rise|rising|fall|falling|moderate|easing)\b", r"\baffordab", r"\brent(s|al) (rise|fall|moderate)\b"],
    "market_supply":        [r"\bsupply\b", r"\bpipeline\b", r"\blaunch(ed|es)\b", r"\bunits?\b", r"\bflat(s)?\b", r"\bhousing supply\b"],
    "demand_trends":        [r"\bdemand\b", r"\bballot\b", r"\bapplications?\b", r"\bstrong take-up\b"],
    "government_policies":  [r"\bABSD\b|\bTDSR\b|\bLTV\b|\bBTO\b|\bGLS\b|\bcooling measures\b|\bHFE\b|\bPHG\b|\bPLPH\b|\bPlus Model\b"],
    "infrastructure_development":[r"\bcheckpoint\b|\bmrt\b|\binterchange\b|\btransport\b|\broad\b|\bbridge\b|\bpark\b|\bhub\b|\bport\b"],
    "economic_factors":     [r"\bunemployment\b|\binflation\b|\binterest rate(s)?\b|\bmacroprudential\b|\bglobal economic\b"],
}
POLARITY_TO_MARKET = {"positive":"bullish", "neutral":"neutral", "negative":"bearish"}
POLARITY_TO_INVEST = {"positive":"optimistic","neutral":"neutral","negative":"pessimistic"}

def _match_any(patterns, text):
    for pat in patterns:
        if re.search(pat, text, flags=re.I):
            return True
    return False

def heuristic_polarity(text: str) -> tuple[str, float, int, list]:
    if not isinstance(text, str) or not text.strip():
        return "neutral", 0.3, 3, []
    t = re.sub(r"\s+", " ", text).strip()
    pos = _match_any(POS_CUES, t)
    neg = _match_any(NEG_CUES, t)
    neu = _match_any(NEU_CUES, t)
    drivers = []
    if pos: drivers.append("positive cues found")
    if neg: drivers.append("negative cues found")
    if neu: drivers.append("neutral announcer cues")
    if pos and not neg:
        pol, conf, inten = "positive", 0.8, 8
    elif neg and not pos:
        pol, conf, inten = "negative", 0.8, 7
    elif pos and neg:
        pol, conf, inten = "neutral", 0.5, 5
    else:
        pol, conf, inten = "neutral", 0.5 if neu else 0.4, 4 if neu else 3
    return pol, conf, inten, drivers

def heuristic_absa(text: str) -> dict:
    out = {}
    t = text or ""
    for aspect, pats in ASPECT_CUES.items():
        if any(re.search(p, t, flags=re.I) for p in pats):
            pos = _match_any(POS_CUES, t)
            neg = _match_any(NEG_CUES, t)
            if pos and not neg:
                sent, conf = "positive", 0.7
            elif neg and not pos:
                sent, conf = "negative", 0.7
            else:
                sent, conf = "neutral", 0.5
            out[aspect] = {"sentiment": sent, "confidence": conf, "key_phrases": []}
    return out

def seed_heuristics(df: pd.DataFrame) -> pd.DataFrame:
    def is_empty_json(s):
        if not isinstance(s, str): return True
        s = s.strip()
        return (s == "" or s == "{}" or s.lower() == "nan")
    df = df.copy()
    df["title"] = df.get("title","").astype(str).fillna("")
    df["clean_text"] = df.get("clean_text", df.get("text","")).astype(str).fillna("")
    df["body"] = (df["title"].fillna("") + "\n\n" + df["clean_text"].fillna("")).str.strip()
    seeds = 0
    for i, row in df.iterrows():
        if is_empty_json(row.get("property_sentiment","{}")):
            pol, conf, inten, drivers = heuristic_polarity(row["body"])
            market = POLARITY_TO_MARKET[pol]
            invest = POLARITY_TO_INVEST[pol]
            ps = {
                "overall_sentiment": {"value": pol, "confidence_score": conf},
                "property_market_sentiment": {"value": market, "confidence_score": max(conf-0.05, 0.0)},
                "investor_sentiment": {"value": invest, "confidence_score": max(conf-0.1, 0.0)},
                "key_sentiment_drivers": drivers,
                "sentiment_intensity": inten
            }
            df.at[i, "property_sentiment"] = json.dumps(ps, ensure_ascii=False)
            seeds += 1
        if is_empty_json(row.get("aspect_based_sentiment","{}")):
            absa = heuristic_absa(row["body"])
            if absa:
                df.at[i, "aspect_based_sentiment"] = json.dumps(absa, ensure_ascii=False)
    print(f"[HEURISTIC] Seeded property_sentiment for {seeds} rows.")
    return df

# ---- Flatteners & Flags ----
ASPECT_MAP = {
    "price_affordability": "affordability_sentiment",
    "market_supply": "supply_sentiment",
    "demand_trends": "demand_sentiment",
    "government_policies": "policy_sentiment",
    "infrastructure_development": "infrastructure_sentiment",
    "economic_factors": "economic_sentiment",
    "location": "location_sentiment",
    "lifestyle": "lifestyle_sentiment",
}

def flatten_property(df: pd.DataFrame) -> pd.DataFrame:
    out = {k: [] for k in [
        "overall_sentiment","overall_score","price_sentiment","price_score",
        "investor_sentiment","investor_score","sentiment_intensity","sentiment_drivers"
    ]}
    for s in df["property_sentiment"]:
        d = s if isinstance(s, dict) else safe_json_load(s)
        if not d:
            for k in out: out[k].append(None if k!="sentiment_drivers" else [])
            continue
        ov = d.get("overall_sentiment");        out["overall_sentiment"].append(pick_val(ov)); out["overall_score"].append(pick_score(ov))
        pm = d.get("property_market_sentiment");out["price_sentiment"].append(pick_val(pm));  out["price_score"].append(pick_score(pm))
        inv= d.get("investor_sentiment");       out["investor_sentiment"].append(pick_val(inv)); out["investor_score"].append(pick_score(inv))
        out["sentiment_intensity"].append(d.get("sentiment_intensity"))
        out["sentiment_drivers"].append(d.get("key_sentiment_drivers", []))
    for k,v in out.items(): df[k]=v
    return df

def _write_aspect(df, i, asp, payload):
    if not isinstance(asp, str): return
    key = asp.strip().lower()
    out = ASPECT_MAP.get(key)
    if not out: return
    sent_col = out
    score_col= out.replace("_sentiment","_score")
    if isinstance(payload, dict):
        df.at[i, sent_col]  = pick_val(payload)
        df.at[i, score_col] = pick_score(payload)
    else:
        df.at[i, sent_col]  = pick_val({"value": payload})

def flatten_aspects(df: pd.DataFrame) -> pd.DataFrame:
    for out in set(ASPECT_MAP.values()):
        if out not in df.columns: df[out]=None
        sc = out.replace("_sentiment","_score")
        if sc not in df.columns: df[sc]=None
    for i, raw in enumerate(df["aspect_based_sentiment"]):
        d = raw if isinstance(raw, dict) else safe_json_load(raw)
        if not d: continue
        if isinstance(d, dict):
            for key in ("analysis","aspect_sentiment_analysis","results"):
                nest = d.get(key)
                if isinstance(nest, list):
                    for item in nest:
                        if isinstance(item, dict):
                            _write_aspect(df, i, item.get("aspect"),
                                          {"value": item.get("sentiment"), "confidence": item.get("confidence")})
                elif isinstance(nest, dict):
                    for asp, payload in nest.items():
                        _write_aspect(df, i, asp, payload)
            for asp, payload in d.items():
                if asp in ("analysis","aspect_sentiment_analysis","results"): continue
                _write_aspect(df, i, asp, payload)
        elif isinstance(d, list):
            for item in d:
                if isinstance(item, dict) and "aspect" in item:
                    _write_aspect(df, i, item.get("aspect"),
                                  {"value": item.get("sentiment"), "confidence": item.get("confidence")})
    return df

def derive_fields(df: pd.DataFrame) -> pd.DataFrame:
    hay = (df.get("title","").astype(str) + " " + df.get("clean_text","").astype(str)).str.lower()
    df["policy_mentioned"] = hay.apply(policy_flag)

    def _choose_loc(row):
        explicit = row.get("primary_location")
        if explicit: return explicit
        fallback = first_location(f"{row.get('title','')} {row.get('clean_text','')}")
        return fallback or "Singapore (national)"
    df["location"] = df.apply(_choose_loc, axis=1)

    ordered = [
        ("policy_sentiment","Policy"),("price_sentiment","Price/Affordability"),("demand_sentiment","Demand"),
        ("supply_sentiment","Supply"),("infrastructure_sentiment","Infrastructure"),
        ("economic_sentiment","Economy"),("location_sentiment","Location"),
    ]
    def choose_aspect(row):
        for col,label in ordered:
            v = str(row.get(col,"") or "").lower()
            if v and v not in ("neutral","unknown","not_applicable","not mentioned","nan"):
                return label
        return "General"
    df["aspect"] = df.apply(choose_aspect, axis=1)
    return df

def backfill_and_flags(df: pd.DataFrame) -> pd.DataFrame:
    for c in ["lifestyle_sentiment","thread_url","forum_name","prop_topic_tags","entity","entity_labels","location"]:
        if c not in df.columns:
            df[c] = None

    if "timestamp" in df.columns:
        df["date"] = pd.to_datetime(df["timestamp"], errors="coerce")
        try:
            df["temporal_bucket"] = df["date"].dt.to_period("W").astype(str)
        except Exception:
            df["temporal_bucket"] = None

    df["flag_Sentiment"] = df[[
        "overall_sentiment","price_sentiment","policy_sentiment","affordability_sentiment","location_sentiment"
    ]].notna().any(axis=1)

    df["flag_Aspect"] = df[[
        "policy_sentiment","demand_sentiment","supply_sentiment",
        "infrastructure_sentiment","economic_sentiment","price_sentiment"
    ]].notna().any(axis=1)

    df["flag_Entity"] = df["entity"].astype(str).str.len().fillna(0).astype(int) > 0
    df["flag_Location"] = df["location"].notna()
    df["flag_Date"] = df["date"].notna() if "date" in df.columns else False

    # ✅ FIXED Singlish flag — no capturing groups warning
    pattern = r"\b(?:" + "|".join([re.escape(x) for x in list(SINGLISH_TERMS)[:50]]) + r")\b"
    df["flag_Singlish"] = (
        df.get("singlish_cultural_context","")
          .astype(str)
          .str.contains(pattern, case=False, regex=True, na=False)
    )

    df["flag_Policy"] = df["policy_mentioned"].fillna(False)
    return df




In [ ]:

# ============================================
# CELL 11: Labeling runner + Regex/EntityRuler + Policy Lexicon
# ============================================
def build_body_columns(df: pd.DataFrame) -> pd.DataFrame:
    title = df.get("metadata.title", df.get("title",""))
    text  = df.get("cleaned_text", df.get("text",""))
    df = df.copy()
    df["title"] = title.apply(clean_text)
    df["clean_text"] = text.apply(clean_text)
    df["body"] = (df["title"].fillna("") + "\n\n" + df["clean_text"].fillna("")).str.strip()
    return df

def label_dataframe(df: pd.DataFrame, model: str = MODEL, batch_size: int = BATCH_SIZE, max_records: int = MAX_RECORDS) -> pd.DataFrame:
    df = build_body_columns(df)
    if AGENCY_ONLY:
        df = df[df.get("agency","").astype(str).str.upper() == AGENCY_ONLY.upper()].copy()
        logger.info(f"Filtered to agency={AGENCY_ONLY.upper()}: {len(df)} rows")
    if max_records and max_records > 0:
        df = df.head(max_records).copy()
        logger.info(f"Limiting to first {len(df)} rows for testing")

    client = GPTLabeler(GPTConfig(model=model))
    labeled_rows = []
    n = len(df)
    for i in range(0, n, batch_size):
        batch = df.iloc[i:i+batch_size]
        for idx, row in batch.iterrows():
            body = row["body"]
            try:
                lab = client.label_one(body)
            except Exception as e:
                lab = {"error": str(e)}
            labeled_rows.append({
                "id": row.get("id", idx),
                "agency": row.get("agency"),
                "source_file": row.get("source_file"),
                "timestamp": row.get("timestamp"),
                "url": row.get("url"),
                "language": row.get("language"),
                "title": row.get("title"),
                "clean_text": row.get("clean_text"),
                "property_sentiment": json.dumps(lab.get("property_sentiment", {}), ensure_ascii=False),
                "aspect_based_sentiment": json.dumps(lab.get("aspect_based_sentiment", {}), ensure_ascii=False),
                "singlish_cultural_context": json.dumps(lab.get("singlish_cultural", {}), ensure_ascii=False),
                "named_entities": json.dumps(lab.get("named_entities", {}), ensure_ascii=False),
                "policy_impact": json.dumps(lab.get("policy_impact", {}), ensure_ascii=False),
                "location_impact": json.dumps(lab.get("location_impact", {}), ensure_ascii=False),
                "domain_expertise": json.dumps(lab.get("domain_expertise", {}), ensure_ascii=False),
                "labeling_timestamp": now_iso(),
            })
        time.sleep(0.25)
        logger.info(f"Labeled {min(i+batch_size, n)}/{n}")
    return pd.DataFrame(labeled_rows)

def enrich_labeled(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = flatten_property(df)
    df = flatten_aspects(df)
    pairs = compile_regex_flags(REGEX_JSONL)
    df = apply_regex_flags(df, pairs)
    df = entityruler_entities(df, ENTITYRULER)
    df = enrich_with_policy_lexicon(df)
    df = derive_fields(df)
    df = backfill_and_flags(df)
    return df

def save_split_by_agency(df: pd.DataFrame, outdir: str, suffix: str):
    out = Path(outdir); out.mkdir(parents=True, exist_ok=True)
    agencies = sorted([a for a in df.get("agency","").dropna().unique().tolist() if a])
    paths = []
    for ag in agencies:
        sub = df[df["agency"]==ag].copy()
        p = out / f"{ag}_{suffix}.csv"
        sub.to_csv(p, index=False)
        paths.append(p)
    all_path = out / f"ALL_{suffix}.csv"
    df.to_csv(all_path, index=False)
    paths.append(all_path)
    return paths

In [ ]:
# ============================================
# CELL 12: Build Combined from RAW (auto) → Clean → Seed → Label → Enrich → Save
# ============================================
# 0) RAW discovery + fallback
auto_raw = find_best_raw_root(BASE)
use_raw = auto_raw if auto_raw else RAW_BASE
print(f"[RAW SELECTED] Using: {use_raw if auto_raw else RAW_BASE}")

try:
    df_combined = normalize_raw_to_combined(use_raw, RAW_COMBINED)
    print(f"[OK] Combined from RAW: {len(df_combined)} rows")
except FileNotFoundError:
    print(f"[INFO] RAW not found or empty. Falling back to processed CSVs under: {PROCESSED_BASE}")
    df_combined = build_combined_from_processed(PROCESSED_BASE, RAW_COMBINED)

# 1) Clean noise BEFORE labeling
df_combined = apply_cleaner(df_combined)

# 2) Heuristic seed for property_sentiment/ABSA (cheap prefill)
df_seeded = seed_heuristics(df_combined)

# 3) Label (OpenAI)
df_labeled = label_dataframe(df_seeded, model=MODEL, batch_size=BATCH_SIZE, max_records=MAX_RECORDS)
labeled_paths = save_split_by_agency(df_labeled, OUTPUT_DIR, suffix="labeled")
print("Wrote labeled CSVs:")
for p in labeled_paths: print("  -", p)

# 4) Enrich/flatten for dashboard + policy/locations/entities/flags
df_enriched = enrich_labeled(df_labeled)
enriched_paths = save_split_by_agency(df_enriched, OUTPUT_DIR, suffix="labeled_enriched")
print("Wrote enriched CSVs:")
for p in enriched_paths: print("  -", p)

print("✅ Done.")


[RAW SEARCH] Candidates & counts:
  - /content/drive/MyDrive/PropInsight/raw : 24
  - /content/drive/MyDrive/PropInsight/raw/government_websites : 23
  - /content/drive/MyDrive/PropInsight/data/raw : 0
  - /content/drive/MyDrive/PropInsight/government_websites : 0
  - /content/drive/MyDrive/PropInsight/downloads : 0
  - /content/drive/MyDrive/PropInsight/scraped : 0
[RAW SELECTED] Using: /content/drive/MyDrive/PropInsight/raw
[OK] Combined from RAW: 3707 rows
[HEURISTIC] Seeded property_sentiment for 3707 rows.
Wrote labeled CSVs:
  - /content/drive/MyDrive/PropInsight/labeled/BCA_labeled.csv
  - /content/drive/MyDrive/PropInsight/labeled/HDB_labeled.csv
  - /content/drive/MyDrive/PropInsight/labeled/MAS_labeled.csv
  - /content/drive/MyDrive/PropInsight/labeled/MND_labeled.csv
  - /content/drive/MyDrive/PropInsight/labeled/SFA_labeled.csv
  - /content/drive/MyDrive/PropInsight/labeled/SLA_labeled.csv
  - /content/drive/MyDrive/PropInsight/labeled/UNK_labeled.csv
  - /content/drive/MyD